01. Setup

In [8]:
# Instalação do Java e Spark
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

# Variáveis de ambiente
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Pacote Iceberg
!wget -q https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar -P /content/spark-3.5.0-bin-hadoop3/jars/

import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

# Sessão Spark
spark = SparkSession.builder \
    .appName("IcebergWithSpark") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.6.1,org.postgresql:postgresql:42.3.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "/content/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

# Diretório do Warehouse
!mkdir -p /content/warehouse

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [9]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [10]:
# Criamos a tabela vendas
spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.vendas_iceberg")

spark.sql("""
    CREATE TABLE IF NOT EXISTS hadoop_catalog.default.vendas_iceberg (
        id_vendas INT,
        id_veiculos INT,
        id_concessionarias INT,
        id_vendedores INT,
        id_clientes INT,
        valor_pago DOUBLE,
        data_venda TIMESTAMP,
        data_inclusao TIMESTAMP,
        data_atualizacao TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (days(data_venda))
""")

DataFrame[]

In [11]:
# Credenciais do PostgreSQL
jdbc_hostname = "159.223.187.110"
jdbc_port = 5432
jdbc_database = "novadrive"
jdbc_username = "etlreadonly"
jdbc_password = "novadrive376A@"

# URL JDBC de conexão
jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"

# Propriedades de conexão JDBC
connection_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "org.postgresql.Driver"
}

In [12]:
def read_postgres_data(last_run_timestamp):
    query = f"""
        (SELECT *
         FROM vendas
         WHERE data_inclusao > '{last_run_timestamp}' OR data_atualizacao > '{last_run_timestamp}'
        ) AS vendas_subquery
    """
    df = spark.read.jdbc(
        url=jdbc_url,
        table=query,
        properties=connection_properties
    )
    return df

In [13]:
# Timestamp apenas para a primeira execução

# Data atual
current_date = datetime.now()

# Data de seis meses atrás
last_run_timestamp = current_date - relativedelta(months=6)

# Formatar como string no formato desejado
last_run_timestamp_str = last_run_timestamp.strftime("%Y-%m-%d %H:%M:%S")

print("Data seis meses atrás:", last_run_timestamp_str)

Data seis meses atrás: 2025-10-07 20:55:18


In [ ]:
# Ler dados do PostgreSQL
df_postgres = read_postgres_data(last_run_timestamp)

# Exibir os dados lidos
df_postgres.show()

In [ ]:
# Criar uma visão temporária
df_postgres.createOrReplaceTempView("vendas_updates")

# Executar o MERGE INTO
spark.sql("""
    MERGE INTO hadoop_catalog.default.vendas_iceberg AS target
    USING vendas_updates AS source
    ON target.id_vendas = source.id_vendas
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")

In [ ]:
row_count = spark.sql("SELECT COUNT(*) FROM hadoop_catalog.default.vendas_iceberg").collect()[0][0]
print(f"Total de linhas na tabela vendas_iceberg: {row_count}")
#26214

In [ ]:
# Atualizar o timestamp da última execução
last_run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Processo concluído. Novo timestamp da última execução: {last_run_timestamp}")